# 🗄️ Database Comparison — `DBold.duckdb`  vs  `DBnew.duckdb`

**Goal:** clearly surface *every* difference between the old multi-plate milling database
and the new single-plate (Platte 40, 40 slots) ET200 database, so the project can safely
migrate onto the new data.

**Context / expectations**
- New DB = **1 plate, 40 slots** (old had 7 plates).
- Oscilloscope is now called **ET200**.
- HF signals that used to live in wide columns are now delivered as their own rows/origin.
- **No real chatter** in the new run — the spindle only moved, it did not mill. So any
  load/torque/chatter difference is *expected*, not a data error.

We reuse the existing **`DuckDBLoader`** infrastructure (`src/loader.py`) — two loaders,
one per DB — instead of the `provider` singleton, so both databases stay open side-by-side.

> Legend used below: 🟢 same · 🟡 changed / renamed · 🔴 removed · 🆕 added · ⚠️ breaks current code

In [ ]:
# --- Setup: open BOTH databases through the existing loader infrastructure ---
import sys, re
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# make src importable without the full plotly preamble (keeps this notebook self-contained)
project_root = Path.cwd().resolve().parents[0]   # .../oxford_notebook
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.loader import DuckDBLoader

DATA = project_root / "data"
old = DuckDBLoader(DATA / "DBold.duckdb", read_only=True)
new = DuckDBLoader(DATA / "DBnew.duckdb", read_only=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

def show(title, obj):
    display(Markdown(f"**{title}**"))
    display(obj)

print(f"OLD  db={old.db_path.name:14s} table={old.table_name}")
print(f"NEW  db={new.db_path.name:14s} table={new.table_name}")

## 1 · Table name & size

The whole table was renamed and is ~5.6× smaller (shorter, non-milling run).
`DuckDBLoader` auto-detects the single table, so the rename is transparent to the loader.

In [ ]:
def table_overview(loader):
    n = loader.con.execute(f"SELECT COUNT(*) FROM {loader.table_name}").fetchone()[0]
    ncols = len(loader._valid_cols)
    return {"table": loader.table_name, "rows": n, "columns": ncols}

overview = pd.DataFrame([table_overview(old), table_overview(new)], index=["OLD", "NEW"])
show("Table overview", overview)

## 2 · Column schema diff  (added / removed / type-changed)

The core structural change. Note especially:
- `Platte` **VARCHAR → INTEGER**, `Nut` **DOUBLE → INTEGER**, `Time` **TIMESTAMP_NS → TIMESTAMP**
  — the project should inherit these cleaner integer types.
- The wide **`HFBlockEvent_*`** family is gone (that data moved into an `HF_Event` origin — see §6).

In [ ]:
def schema_map(loader):
    return {r[0]: r[1] for r in loader.con.execute(f"DESCRIBE {loader.table_name}").fetchall()}

so, sn = schema_map(old), schema_map(new)
old_cols, new_cols = set(so), set(sn)

added   = sorted(new_cols - old_cols)
removed = sorted(old_cols - new_cols)
changed = sorted(c for c in (old_cols & new_cols) if so[c] != sn[c])

show("🆕 Columns ADDED in new (%d)" % len(added),
     pd.DataFrame({"column": added, "new_type": [sn[c] for c in added]}))
show("🔴 Columns REMOVED in new (%d)" % len(removed),
     pd.DataFrame({"column": removed, "old_type": [so[c] for c in removed]}))
show("🟡 Columns with TYPE CHANGE (%d)" % len(changed),
     pd.DataFrame({"column": changed,
                   "old_type": [so[c] for c in changed],
                   "new_type": [sn[c] for c in changed]}))

## 3 · Structural axes — plates, slots, data origins

Confirms the headline facts: **1 plate (40), 40 slots**, and the origin rename
`Oscilloscope → ET200_Data` plus a brand-new `HF_Event` origin.

In [ ]:
def axis_vals(loader, col):
    return [r[0] for r in loader.con.execute(
        f"SELECT DISTINCT {col} FROM {loader.table_name} WHERE {col} IS NOT NULL ORDER BY 1"
    ).fetchall()]

for col in ["Platte", "Nut", "DataOrigin"]:
    ov, nv = axis_vals(old, col), axis_vals(new, col)
    show(f"{col}: OLD ({len(ov)}) vs NEW ({len(nv)})",
         pd.DataFrame({"OLD": [ov], "NEW": [nv],
                       "only_in_OLD": [sorted(set(map(str,ov)) - set(map(str,nv)))],
                       "only_in_NEW": [sorted(set(map(str,nv)) - set(map(str,ov)))]}).T
         .rename(columns={0: col}))

## 4 · Signal inventory diff

How many signals are shared, dropped, or new — overall and **per DataOrigin**
(so you can see LF telemetry expanding and the new HF_Event / ET200 origins).

In [ ]:
def signal_set(loader, origin=None):
    q = f"SELECT DISTINCT Signal FROM {loader.table_name} WHERE Signal IS NOT NULL"
    if origin:
        q += f" AND DataOrigin = '{origin}'"
    return {r[0] for r in loader.con.execute(q).fetchall()}

osig, nsig = signal_set(old), signal_set(new)
shared, old_only, new_only = osig & nsig, osig - nsig, nsig - osig

show("Signal counts", pd.DataFrame({
    "count": [len(osig), len(nsig), len(shared), len(old_only), len(new_only)]},
    index=["OLD total", "NEW total", "🟢 shared", "🔴 only in OLD", "🆕 only in NEW"]))

show("🔴 Signals only in OLD", pd.DataFrame({"signal": sorted(old_only)}))
show("🆕 Signals only in NEW", pd.DataFrame({"signal": sorted(new_only)}))

In [ ]:
# Per-origin signal counts (union of origins across both DBs)
origins = sorted(set(axis_vals(old, "DataOrigin")) | set(axis_vals(new, "DataOrigin")))
rows = []
for o in origins:
    oc, nc = signal_set(old, o), signal_set(new, o)
    rows.append({"DataOrigin": o, "OLD_signals": len(oc), "NEW_signals": len(nc),
                 "shared": len(oc & nc), "only_OLD": len(oc - nc), "only_NEW": len(nc - oc)})
show("Signals per DataOrigin", pd.DataFrame(rows).set_index("DataOrigin"))

## 5 · Rename detector — separating *renames* from *real* additions

Many "new" signals are the **same signal renamed**. Two rename patterns exist:
1. **Whitespace**: old `actFeedRate[u1,1]` → new `actFeedRate[u1, 1]` (space after comma).
2. **Channel-index dropped**: old `actToolIdent[u1,1]` → new `actToolIdent[u1]`.

After accounting for these, the list of *genuinely* new signals is much shorter.

In [ ]:
def norm_ws(s):      # collapse whitespace after commas
    return re.sub(r",\s*", ",", s)

def stem(s):         # drop a trailing ",<digits>" channel index inside the last [...]
    return re.sub(r",\d+\]$", "]", norm_ws(s))

# 1) whitespace-only renames
o_ws = {norm_ws(s): s for s in osig}
n_ws = {norm_ws(s): s for s in nsig}
ws_renames = [(o_ws[k], n_ws[k]) for k in (set(o_ws) & set(n_ws))
              if o_ws[k] != n_ws[k]]

# 2) channel-drop renames among still-unmatched signals
still_old = {s for s in old_only if norm_ws(s) not in n_ws}
still_new = {s for s in new_only if norm_ws(s) not in o_ws}
o_stem = {stem(s): s for s in still_old}
n_stem = {stem(s): s for s in still_new}
chan_renames = [(o_stem[k], n_stem[k]) for k in (set(o_stem) & set(n_stem))]

show("🟡 Whitespace renames (old ➜ new)",
     pd.DataFrame(ws_renames, columns=["OLD", "NEW"]) if ws_renames else "none")
show("🟡 Channel-index-dropped renames (old ➜ new)",
     pd.DataFrame(chan_renames, columns=["OLD", "NEW"]) if chan_renames else "none")

renamed_new = {n for _, n in ws_renames} | {n for _, n in chan_renames}
truly_new = sorted(new_only - renamed_new)
show(f"🆕 TRULY new signals after removing renames ({len(truly_new)})",
     pd.DataFrame({"signal": truly_new}))

## 6 · Where did the HF block-event data go?

In the old DB the block-event info was wide columns (`HFBlockEvent_Channel`,
`HFBlockEvent_SeekOffset`, `HFBlockEvent_IpoGC`, …). In the new DB it is delivered as a
dedicated **`HF_Event`** DataOrigin whose signals mirror those fields.

In [ ]:
old_hf_cols = sorted(c for c in old_cols if c.startswith("HFBlockEvent_"))
new_hf_event_signals = sorted(signal_set(new, "HF_Event"))
show("🔴 OLD wide HFBlockEvent_* columns", pd.DataFrame({"old_column": old_hf_cols}))
show("🆕 NEW HF_Event origin signals", pd.DataFrame({"new_signal": new_hf_event_signals}))

# ET200 / Oscilloscope mapping
show("Oscilloscope ➜ ET200 payload",
     pd.DataFrame({"OLD 'Oscilloscope' signals": [sorted(signal_set(old, "Oscilloscope"))],
                   "NEW 'ET200_Data' signals": [sorted(signal_set(new, "ET200_Data"))]}).T
     .rename(columns={0: "signals"}))

## 7 · ⚠️ Loader-dependency audit — what breaks on the new schema

`loader.py` hardcodes specific signal names and columns. Here we check each one against the
**new** schema. ❌ rows are silent failures (no error, just NULL/empty results) that must be
fixed before the project runs on the new DB.

In [ ]:
# signals hardcoded inside loader.py (slot_metadata_summary + slot_chatter_cases_summary)
loader_signals = {
    "R310  (chatter flag)":            "R310",
    "R319  (Nut_ID)":                  "R319",
    "R321  (X position)":              "R321",
    "R330  (Drehzahl)":                "R330",
    "actToolRadius[u1] (Werkzeugradius)": "actToolRadius[u1]",
    "actToolIdent[u1,1] (Werkzeug)":   "actToolIdent[u1,1]",
}
sig_rows = []
for label, s in loader_signals.items():
    in_new = s in nsig
    note = ""
    if not in_new:
        cand = [x for x in nsig if stem(x) == stem(s) or norm_ws(x) == norm_ws(s)]
        note = f"renamed ➜ {cand[0]}" if cand else "missing entirely"
    sig_rows.append({"loader signal": label, "raw": s,
                     "in_new": "✅" if in_new else "❌", "note": note})
show("Hardcoded SIGNALS used by loader", pd.DataFrame(sig_rows))

# columns referenced by get_axiswise_plot_df / get_data_df
loader_cols = ["Platte","Nut","Time","Duration_Seconds","WCS_Y_mm","Axis","Signal","Value",
               "Value_String","DataOrigin","Groupname","Unit","Description",
               "HFBlockEvent_GCode","HFProbeCounter"]
col_rows = [{"column": c, "in_new": "✅" if c in new_cols else "❌"} for c in loader_cols]
show("Columns referenced by loader (get_data_df / get_axiswise_plot_df)",
     pd.DataFrame(col_rows))

## 8 · Value / statistics comparison for shared signals

For signals present in both DBs we compare `count / min / max / mean`. Signals whose physics
depends on cutting (LOAD, TORQUE, CURRENT, POWER, contour deviation) are **expected to differ**
because the new run did not mill — they are flagged so they aren't mistaken for data errors.
We also inspect the `R310` chatter flag. **⚠️ Note:** `R310=1` *is* present in the new DB
(51 rows, 15 slots) even though the run was non-milling — see §17 for this discrepancy.

In [ ]:
def stats(loader, signals):
    placeholders = ",".join("?" * len(signals))
    q = ("SELECT Signal, COUNT(*) n, MIN(Value) vmin, MAX(Value) vmax, AVG(Value) vavg "
         f"FROM {loader.table_name} "
         f"WHERE Signal IN ({placeholders}) GROUP BY Signal")
    return loader.query_df(q, list(signals)).set_index("Signal")

shared_sorted = sorted(shared)
so_stats = stats(old, shared_sorted).add_prefix("OLD_")
sn_stats = stats(new, shared_sorted).add_prefix("NEW_")
cmp = so_stats.join(sn_stats, how="outer")

MILLING = ("LOAD", "TORQUE", "CURRENT", "POWER", "CONT_DEV", "CTRL_DIFF")
cmp["milling_dependent"] = ["⚙️ expected-diff" if any(k in s for k in MILLING) else ""
                            for s in cmp.index]
show(f"Shared-signal statistics ({len(cmp)} signals)", cmp)

In [ ]:
# Chatter flag R310: how many rows / slots carry each value?
def r310_breakdown(loader):
    return loader.query_df(
        "SELECT Value, COUNT(*) AS n, COUNT(DISTINCT Nut) AS slots "
        f"FROM {loader.table_name} WHERE Signal='R310' GROUP BY Value ORDER BY Value")

show("R310 (chatter flag) value distribution — OLD", r310_breakdown(old))
show("R310 (chatter flag) value distribution — NEW", r310_breakdown(new))

## 9 · Migration summary — what the project should inherit

A consolidated decision table of the changes that require action in `loader.py` / downstream
code before switching to the new data.

In [ ]:
summary = pd.DataFrame([
    ["Table name", "my_table ➜ cleaned_data", "🟢 none — DuckDBLoader auto-detects the single table"],
    ["Platte type", "VARCHAR ➜ INTEGER", "🟡 inherit int; drop any str comparison on Platte"],
    ["Nut type", "DOUBLE ➜ INTEGER", "🟡 inherit int"],
    ["Time type", "TIMESTAMP_NS ➜ TIMESTAMP", "🟡 verify datetime parsing still fine"],
    ["HFBlockEvent_* columns", "removed ➜ HF_Event origin rows", "🔴 stop reading wide cols; read HF_Event signals"],
    ["SensorType column", "removed", "🔴 no longer available to tag Accelerometer"],
    ["actToolIdent[u1,1]", "renamed ➜ actToolIdent[u1]", "⚠️ FIX slot_metadata_summary — else Werkzeug=NULL"],
    ["Signal index spacing", "[u1,1] ➜ [u1, 1]", "⚠️ make signal names schema-adaptive"],
    ["Oscilloscope origin", "renamed ➜ ET200_Data (now populated)", "🟡 update origin name / configs"],
    ["Chatter (R310=1)", "absent in new run", "🟢 expected — no milling; not a data error"],
], columns=["Change", "Old ➜ New", "Action"])
show("Migration decision table", summary)

---
# 🔬 Extended checks (§10–§16)

The sections above compare *schemas*. These sections stress-test whether the **existing
downstream infrastructure** (per-slot loops, `slot_metadata_summary`, heatmaps keyed on
`WCS_Y_mm`, HF/FFT work, axis labels, joins) actually holds on the new data.
`NEW_PLATE = 40`; one old plate is used as a reference where a like-for-like view helps.

In [ ]:
NEW_PLATE = new.list_plates()[0]           # 40
OLD_REF_PLATE = old.list_plates()[0]       # reference plate from old DB
print("NEW_PLATE =", NEW_PLATE, "| OLD_REF_PLATE =", OLD_REF_PLATE)

## 10 · Per-slot completeness (new DB)

Every loop over `provider.slots()` assumes all 40 slots exist and carry the expected origins.
Here we count rows / signals / origins per slot and **flag any slot missing an origin**.

In [ ]:
def slot_coverage(loader, plate):
    q = ("SELECT Nut, COUNT(*) AS rows, COUNT(DISTINCT Signal) AS n_signals, "
         "COUNT(DISTINCT DataOrigin) AS n_origins "
         f"FROM {loader.table_name} WHERE Platte = ? AND Nut IS NOT NULL "
         "GROUP BY Nut ORDER BY Nut")
    return loader.query_df(q, [plate])

cov = slot_coverage(new, NEW_PLATE)
show(f"Per-slot coverage — NEW plate {NEW_PLATE} ({len(cov)} slots)", cov)
print("slots present:", len(cov), "| expected 40 ->", "✅" if len(cov) == 40 else "❌ MISSING SLOTS")

# origin presence matrix (slot x origin)
mat = new.query_df(
    "SELECT Nut, DataOrigin, COUNT(*) AS n "
    f"FROM {new.table_name} WHERE Nut IS NOT NULL GROUP BY Nut, DataOrigin")
pivot = mat.pivot(index="Nut", columns="DataOrigin", values="n").fillna(0).astype(int)
missing = pivot[(pivot == 0).any(axis=1)]
show("Rows per (slot × origin) — NEW", pivot)
show("⚠️ Slots missing at least one origin", missing if len(missing) else "none — every slot has every origin ✅")

## 11 · Metadata-signal population per slot

§7 showed the metadata signals *exist*. Here we check they are **populated with non-null `Value`
for every slot** — otherwise `slot_metadata_summary()` returns NULLs. We also run the real loader
method to expose the `actToolIdent[u1,1]` → `Werkzeug` NULL bug on live data.

In [ ]:
meta_sigs = ["R319", "R321", "R330", "actToolRadius[u1]", "R310", "actToolIdent[u1,1]", "actToolIdent[u1]"]
parts = ", ".join(
    f"SUM(CASE WHEN Signal = '{s}' AND Value IS NOT NULL THEN 1 ELSE 0 END) AS \"{s}\""
    for s in meta_sigs)
q = (f"SELECT Nut, {parts} FROM {new.table_name} "
     f"WHERE Platte = {NEW_PLATE} AND Nut IS NOT NULL GROUP BY Nut ORDER BY Nut")
pop = new.query_df(q).set_index("Nut")
show("Non-null Value count per slot for metadata signals — NEW", pop)
zero_cols = [c for c in pop.columns if (pop[c] == 0).all()]
print("metadata signals EMPTY for ALL slots:", zero_cols or "none")

show("Live `loader.slot_metadata_summary(40)` on NEW  (watch the Werkzeug column)",
     new.slot_metadata_summary(NEW_PLATE))

## 12 · `WCS_Y_mm` coverage & range per slot

Every heatmap keys off `WCS_Y_mm`. A non-milling run may not traverse the full slot length, so
the Y-range and null fraction can differ. We compare per-slot range and overall null fraction.

In [ ]:
def wcs_per_slot(loader, plate):
    q = ("SELECT Nut, MIN(WCS_Y_mm) AS y_min, MAX(WCS_Y_mm) AS y_max, "
         "COUNT(WCS_Y_mm) AS n_nonnull, COUNT(*) AS n_rows "
         f"FROM {loader.table_name} WHERE Platte = ? AND Nut IS NOT NULL "
         "GROUP BY Nut ORDER BY Nut")
    df = loader.query_df(q, [plate])
    df["y_span"] = df["y_max"] - df["y_min"]
    df["null_frac"] = 1 - df["n_nonnull"] / df["n_rows"]
    return df

show(f"WCS_Y_mm per slot — NEW plate {NEW_PLATE}", wcs_per_slot(new, NEW_PLATE))
show(f"WCS_Y_mm per slot — OLD ref plate {OLD_REF_PLATE} (first 10)", wcs_per_slot(old, OLD_REF_PLATE).head(10))

def wcs_overall(loader):
    r = loader.query_row(
        f"SELECT COUNT(*), COUNT(WCS_Y_mm), MIN(WCS_Y_mm), MAX(WCS_Y_mm) FROM {loader.table_name}")
    return {"rows": r[0], "nonnull": r[1], "null_frac": round(1 - r[1]/r[0], 4),
            "y_min": r[2], "y_max": r[3]}
show("WCS_Y_mm overall", pd.DataFrame([wcs_overall(old), wcs_overall(new)], index=["OLD", "NEW"]))

## 13 · Sampling rate / temporal density

Any FFT / HF-chatter work depends on the sample rate. We compare declared `SamplingPeriod`
per origin, `Duration_Seconds`, and the **real** median time-delta between consecutive HF samples.

In [ ]:
def sampling_by_origin(loader):
    q = ("SELECT DataOrigin, SamplingPeriod, COUNT(*) AS n "
         f"FROM {loader.table_name} WHERE SamplingPeriod IS NOT NULL "
         "GROUP BY DataOrigin, SamplingPeriod ORDER BY DataOrigin, SamplingPeriod")
    return loader.query_df(q)
show("Declared SamplingPeriod per origin — OLD", sampling_by_origin(old))
show("Declared SamplingPeriod per origin — NEW", sampling_by_origin(new))

def real_dt(loader, signal):
    q = (f"SELECT Time FROM {loader.table_name} WHERE Signal = ? AND DataOrigin = 'HF_Data' "
         "ORDER BY Time LIMIT 20000")
    t = loader.query_df(q, [signal])["Time"]
    if len(t) < 3:
        return None
    dt = pd.to_datetime(t).diff().dt.total_seconds()
    return float(dt.median())

hf_probe = "CURRENT|1"
show("Real median HF Δt (s) for '%s'" % hf_probe,
     pd.DataFrame({"median_dt_s": [real_dt(old, hf_probe), real_dt(new, hf_probe)]},
                  index=["OLD", "NEW"]))

## 14 · Unit / Description / Groupname drift for shared signals

If a shared signal's `Unit` or `Groupname` changed, plot labels silently go wrong. We surface
only the **mismatches**.

In [ ]:
def meta_per_signal(loader, signals):
    placeholders = ",".join("?" * len(signals))
    q = ("SELECT Signal, MIN(Unit) AS unit, MIN(Groupname) AS grp, MIN(Description) AS descr "
         f"FROM {loader.table_name} WHERE Signal IN ({placeholders}) GROUP BY Signal")
    return loader.query_df(q, list(signals)).set_index("Signal")

om = meta_per_signal(old, shared_sorted).add_prefix("OLD_")
nm = meta_per_signal(new, shared_sorted).add_prefix("NEW_")
meta = om.join(nm, how="outer")
unit_diff = meta[meta["OLD_unit"].fillna("") != meta["NEW_unit"].fillna("")][["OLD_unit", "NEW_unit"]]
grp_diff  = meta[meta["OLD_grp"].fillna("")  != meta["NEW_grp"].fillna("")][["OLD_grp", "NEW_grp"]]
show(f"🟡 Unit mismatches ({len(unit_diff)})", unit_diff if len(unit_diff) else "none — units consistent ✅")
show(f"🟡 Groupname mismatches ({len(grp_diff)})", grp_diff if len(grp_diff) else "none — groupnames consistent ✅")

## 15 · Null-density per shared column

A column can exist (§2) yet be mostly empty. We compare the **fraction populated** for every
column present in both DBs.

In [ ]:
shared_cols = sorted(old_cols & new_cols)
def fill_frac(loader, cols):
    total = loader.query_row(f"SELECT COUNT(*) FROM {loader.table_name}")[0]
    sel = ", ".join(f'COUNT("{c}")' for c in cols)
    counts = loader.query_row(f"SELECT {sel} FROM {loader.table_name}")
    return {c: round(n / total, 4) for c, n in zip(cols, counts)}

density = pd.DataFrame({"OLD_filled": fill_frac(old, shared_cols),
                        "NEW_filled": fill_frac(new, shared_cols)})
density["Δ"] = (density["NEW_filled"] - density["OLD_filled"]).round(4)
show("Fraction populated per shared column (0–1)", density.sort_values("Δ"))

## 16 · `Value_Type`, `Value_String`, and the new `Cycle` column

Confirms string-valued signals still populate `Value_String`, and checks whether the new
`Cycle` column changes **row granularity** (i.e. is `(Time, Signal, Nut, DataOrigin)` still a
unique key, or does `Cycle` disambiguate duplicates?).

In [ ]:
def value_type_breakdown(loader):
    return loader.query_df(
        "SELECT Value_Type, COUNT(*) AS n "
        f"FROM {loader.table_name} GROUP BY Value_Type ORDER BY n DESC")
show("Value_Type distribution — OLD", value_type_breakdown(old))
show("Value_Type distribution — NEW", value_type_breakdown(new))

# Value_String usage
for label, loader in [("OLD", old), ("NEW", new)]:
    n = loader.query_row(
        f"SELECT COUNT(*) FROM {loader.table_name} WHERE Value_String IS NOT NULL")[0]
    ex = loader.query_df(
        "SELECT DISTINCT Signal FROM "
        f"{loader.table_name} WHERE Value_String IS NOT NULL LIMIT 8")["Signal"].tolist()
    print(f"{label}: Value_String non-null rows = {n:,} | example signals = {ex}")

# Cycle granularity (new only)
tot = new.query_row(f"SELECT COUNT(*) FROM {new.table_name}")[0]
key_no_cycle = new.query_row(
    "SELECT COUNT(*) FROM (SELECT DISTINCT Time, Signal, Nut, DataOrigin FROM "
    f"{new.table_name})")[0]
key_with_cycle = new.query_row(
    "SELECT COUNT(*) FROM (SELECT DISTINCT Time, Signal, Nut, DataOrigin, Cycle FROM "
    f"{new.table_name})")[0]
cyc = new.query_row(f"SELECT MIN(Cycle), MAX(Cycle), COUNT(DISTINCT Cycle) FROM {new.table_name}")
show("Row-granularity check — NEW", pd.DataFrame({
    "value": [tot, key_no_cycle, key_with_cycle, cyc[0], cyc[1], cyc[2]]},
    index=["total rows", "distinct (Time,Signal,Nut,Origin)",
           "distinct (+Cycle)", "Cycle min", "Cycle max", "Cycle distinct"]))
print("Cycle disambiguates duplicate keys:" ,
      "✅ yes" if key_with_cycle > key_no_cycle else "no — key already unique without Cycle")

---
## 17 · 🔎 Is any data *clearly missing* in DBnew?

The direct answer. We separate three cases:
1. **Signals truly gone** (present in old, no rename match in new).
2. **Columns that exist but are (near-)empty** in new.
3. **Completeness** of the analysis-critical signals & origins (nothing silently absent per slot).

Plus the **R310 chatter discrepancy**: the run was described as non-milling, yet `R310=1` exists.

In [ ]:
# 1) Signals in OLD with no rename match in NEW  (uses norm_ws/stem from §5)
n_norm = {norm_ws(s) for s in nsig}
n_stem = {stem(s)    for s in nsig}
truly_missing = sorted(s for s in old_only
                       if norm_ws(s) not in n_norm and stem(s) not in n_stem)

# context for each missing signal (where it lived, how much data it had in OLD)
if truly_missing:
    ph = ",".join("?" * len(truly_missing))
    ctx = old.query_df(
        "SELECT Signal, MIN(DataOrigin) AS origin, MIN(Groupname) AS groupname, "
        "MIN(Description) AS description, COUNT(*) AS old_rows "
        f"FROM {old.table_name} WHERE Signal IN ({ph}) GROUP BY Signal ORDER BY old_rows DESC",
        truly_missing)
else:
    ctx = pd.DataFrame()
show(f"🔴 Signals TRULY MISSING in new ({len(truly_missing)}) — renames already excluded", ctx)

In [ ]:
# 2) NEW columns that are entirely / nearly empty
new_total = new.query_row(f"SELECT COUNT(*) FROM {new.table_name}")[0]
rows = []
for c in sorted(new_cols):
    nn = new.query_row(f'SELECT COUNT("{c}") FROM {new.table_name}')[0]
    frac = nn / new_total
    if frac < 0.01:
        rows.append({"column": c, "non_null": nn, "filled_%": round(frac * 100, 3)})
show("🟡 NEW columns <1% populated (structurally sparse, not necessarily a problem)",
     pd.DataFrame(rows) if rows else "none")

In [ ]:
# 3) Completeness of analysis-critical signals & origins across ALL slots (multi-plate safe)
plates = new.list_plates()
n_slots_total = new.query_row(f"SELECT COUNT(DISTINCT Nut) FROM {new.table_name} WHERE Nut IS NOT NULL")[0]

crit = {"R319": "Value", "R321": "Value", "R330": "Value", "R310": "Value",
        "actToolRadius[u1]": "Value", "actToolIdent[u1]": "Value_String"}
rows = []
for s, col in crit.items():
    r = new.query_row(
        f"SELECT COUNT(DISTINCT Nut), COUNT(*) FROM {new.table_name} "
        f"WHERE Signal=? AND {col} IS NOT NULL", [s])
    rows.append({"signal": s, "value_col": col, "slots_covered": r[0],
                 "of_slots": n_slots_total, "rows": r[1],
                 "complete": "✅" if r[0] == n_slots_total else "❌"})
show("Analysis-critical signals — per-slot completeness (NEW)", pd.DataFrame(rows))

orows = []
for o in sorted(axis_vals(new, "DataOrigin")):
    r = new.query_row(
        f"SELECT COUNT(DISTINCT Nut), COUNT(DISTINCT Signal), COUNT(*) FROM {new.table_name} "
        "WHERE DataOrigin=?", [o])
    orows.append({"origin": o, "slots_covered": r[0], "of_slots": n_slots_total,
                  "signals": r[1], "rows": r[2],
                  "complete": "✅" if r[0] == n_slots_total else "❌"})
show("DataOrigin — per-slot coverage (NEW)", pd.DataFrame(orows))

In [ ]:
# R310 chatter discrepancy: non-milling run, yet R310=1 exists?
r310 = new.query_df(
    "SELECT Value, COUNT(*) AS rows, COUNT(DISTINCT Nut) AS slots "
    f"FROM {new.table_name} WHERE Signal='R310' GROUP BY Value ORDER BY Value")
show("⚠️ R310 in NEW — chatter flag present despite 'no milling'", r310)
chatter_slots = new.query_df(
    "SELECT DISTINCT Nut FROM "
    f"{new.table_name} WHERE Signal='R310' AND Value=1 ORDER BY Nut")["Nut"].tolist()
print("Slots flagged R310=1 in NEW:", chatter_slots)
print("=> Reconcile with domain owner: likely a PLANNED/label flag, not physically-measured chatter.")

### 17b · Where did `ID`, `SensorType`, `Label` go?

Three old columns were dropped. Are they recoverable in the new system?

| Column | What it held in OLD | In NEW? | Verdict |
|---|---|---|---|
| **`SensorType`** | only `'Accelerometer'`, and **only** on `Oscilloscope` rows (36.9 M) | ⇒ `DataOrigin = 'ET200_Data'` | ✅ **recoverable** — it was redundant; derive from `DataOrigin` |
| **`ID`** | sparse per-signal index (e.g. `R310→'4'`, `R319→'8'`) | no column; `Signal` is the real key | ⚠️ **not carried but redundant** — no functional loss |
| **`Label`** | German friendly names (`Rattererkennung`=R310, `Spindeldrehzahl`=R330, `X-Position Nut`=R321, `Kanal A/B (V)` for scope) | `Description` stays **NULL** for these signals | 🔴 **genuinely lost** — did **not** migrate into `Description` |

In [ ]:
# SensorType: prove it was 100% redundant with the Oscilloscope origin
off = old.query_row(
    f"SELECT COUNT(*) FROM {old.table_name} "
    "WHERE SensorType = 'Accelerometer' AND DataOrigin <> 'Oscilloscope'")[0]
print(f"OLD Accelerometer rows NOT on Oscilloscope origin: {off}  -> "
      f"{'✅ fully derivable from DataOrigin (ET200_Data in new)' if off == 0 else 'NOT purely origin-based'}")

# Label: show the friendly names are gone — OLD Label vs NEW Description for the same signals
key_sigs = ["R310", "R319", "R321", "R330", "actToolRadius[u1]"]
ph = ",".join("?" * len(key_sigs))
old_lab = old.query_df(
    'SELECT Signal, MIN("Label") AS OLD_Label, MIN(Description) AS OLD_Description '
    f"FROM {old.table_name} WHERE Signal IN ({ph}) GROUP BY Signal", key_sigs).set_index("Signal")
new_desc = new.query_df(
    "SELECT Signal, MIN(Description) AS NEW_Description, MIN(Groupname) AS NEW_Groupname "
    f"FROM {new.table_name} WHERE Signal IN ({ph}) GROUP BY Signal", key_sigs).set_index("Signal")
show("Friendly-name provenance: OLD `Label` ➜ NEW `Description` (NULL = lost)",
     old_lab.join(new_desc, how="outer"))

# ID: the old per-signal index mapping (redundant with Signal)
show("OLD `ID` → Signal mapping (redundant; Signal is the identity)",
     old.query_df('SELECT DISTINCT "ID", Signal FROM '
                  f'{old.table_name} WHERE "ID" IS NOT NULL ORDER BY TRY_CAST("ID" AS INT)'))
print("⚠️ Code impact: data_processing.extract_unique_signal_values() selects the 'Label' column "
      "-> KeyError on the new schema. Needs a static Signal→label lookup or drop Label.")

### 17c · Decision — labels stay **German**, held in code

**Decision:** preserve the old `Label` strings **in German** and keep them as a static
`Signal → name` map in `src/schema_map.py` (not in the DB). Rationale:
- The codebase & domain are German (`Nut`, `Platte`, `Werkzeug`, German docstrings) — these are the
  established user-facing display names; changing them is cosmetic churn with no upside.
- `Label` (German, display) and the new `Description` (English, technical SINUMERIK — now populated
  for 160/186 signals) are **different fields**; keep both, don't merge or translate.
- Holding the 11-entry map in code makes it stable across re-cleans and identical on old & new DBs.

The map below is generated **from the old DB** so it is verifiable and reproducible.
`R328` is included for completeness though that signal was itself dropped (§17).

> **Is the labeling important to the data-processing functions? No — it is display-only.**
> The dropped `Label` column is read in **exactly one** function, `extract_unique_signal_values()`
> (an inspection/overview helper), and that function has **no callers** anywhere in `src/`, `viz/`
> or the notebooks. All real analysis — chatter detection, heatmaps, PCA, `slot_metadata_summary`,
> axiswise plots — keys off `Signal / Value / WCS_Y_mm / DataOrigin / Nut / Platte`, all present in
> the new DB. The `"Chatter"-Label` / `tick_labels` references in `visualizer.py` are the R310-derived
> 0/1 classification and plot text, **not** the `Label` column. So `SIGNAL_LABELS` is a **display
> enhancement** (nice German axis/legend names), not a migration blocker.

In [ ]:
# Reproducibly build the German label map straight from the OLD database
lbl = old.query_df(
    'SELECT Signal, MIN("Label") AS label FROM '
    f"{old.table_name} WHERE \"Label\" IS NOT NULL AND Signal IS NOT NULL "
    "GROUP BY Signal ORDER BY Signal")
show("Exact German Signal → Label map (from OLD DB)", lbl)

SIGNAL_LABELS = dict(zip(lbl["Signal"], lbl["label"]))
# scope-channel labels had no Signal in OLD (they tagged Oscilloscope rows) — map by ET200 channel
SCOPE_LABELS = {"dataA1CH1": "Kanal A (V)", "dataA2CH1": "Kanal B (V)"}  # confirm CH assignment w/ owner
print("SIGNAL_LABELS =", SIGNAL_LABELS)
print("SCOPE_LABELS  =", SCOPE_LABELS, "(channel↔A/B assignment to be confirmed with data owner)")

In [ ]:
old.close(); new.close()
print("done — all sections executed, connections closed")

---

# 📋 Final Report — DBold ➜ DBnew

## Executive summary
The new database (`DBnew.duckdb`, table **`cleaned_data`**) is the **same long-form structure**
as the old one (`DBold.duckdb`, table **`my_table`**) but is a **single-plate, non-milling
acquisition**: **1 plate (Platte 40), 40 slots, 17.6 M rows** vs the old 7 plates / 99.9 M rows.
The schema is broadly compatible and **all analysis-critical LF signals (R310, R319, R321, R330,
`actToolRadius[u1]`) are present**, so the project can migrate onto the new data — but **two
silent breakages** and a set of **type/naming changes** must be handled first.

## 1. Structural changes (schema)
| Item | OLD | NEW | Impact |
|---|---|---|---|
| Table name | `my_table` | `cleaned_data` | 🟢 none — loader auto-detects |
| Rows | 99,858,658 | 17,643,463 | shorter run |
| `Platte` | `VARCHAR` (7 values) | `INTEGER` (`40`) | 🟡 inherit int |
| `Nut` | `DOUBLE` | `INTEGER` | 🟡 inherit int |
| `Time` | `TIMESTAMP_NS` | `TIMESTAMP` | 🟡 verify parsing |
| Columns removed | — | 12 dropped | 🔴 see below |
| Columns added | — | 5 added | 🆕 `Cycle, IpoGC, NcCode, NcComment, Nc_variable_name` |

**Removed (12):** `SensorType`, `ID`, `Label`, `HFProbeCounter`, and the wide
`HFBlockEvent_GCode / _Channel / _SeekOffset / _SelectedTool / _ActiveTool / _IpoGC /
_ipoReadError / _laBuf` family.

## 2. Origins & signals
- **DataOrigins:** `{HF_Data, LF_Data, Oscilloscope}` ➜ `{HF_Data, LF_Data, ET200_Data, HF_Event}`
  - `Oscilloscope` ➜ **`ET200_Data`** (renamed **and now populated**: `dataA1CH1, dataA1CH2, dataA2CH1`).
    In the old DB `Oscilloscope` was empty.
  - **`HF_Event`** is new — it carries the block-event data that used to be wide columns
    (`HF_EVENT|Channel, |SeekOffset, |IpoGC, |NcCode, |NcComment, |LaBuf, …`).
- **Signals:** 101 shared · 14 old-only · 85 new-only. After removing pure **renames**
  (whitespace `[u1,1]`→`[u1, 1]` and channel-drop `actToolIdent[u1,1]`→`actToolIdent[u1]`),
  the genuinely new signals are mostly **expanded LF/NC telemetry** (LF grew 22 ➜ 82 signals).

## 3. ⚠️ Silent breakages
| # | Problem | Symptom | Fix |
|---|---|---|---|
| 1 | `slot_metadata_summary()` hardcodes `actToolIdent[u1,1]` | renamed to `actToolIdent[u1]` → **`Werkzeug` all-NULL**, no error | resolve signal name per schema |
| 2 | Signal index spacing `[u1,1]` vs `[u1, 1]` | exact-string matches miss | make signal names schema-adaptive |
| 3 | `SensorType` column dropped | can't tag Accelerometer via column | derive from `DataOrigin='ET200_Data'` (was 100% redundant) |
| 4 | `HFBlockEvent_*` columns dropped | GCode overlays read empty columns | read `HF_Event` origin signals |
| 5 🟡 | `Label` column dropped | `extract_unique_signal_values()` selects `Label` → KeyError — **but the function has no callers**, so it's *latent*, not active | fix only if used: swap `Label` for `label_for(signal)`, or delete the helper |

## 4. Expected differences (NOT data errors)
The new run **did not mill** (spindle only moving), so:
- **Load/torque/current/power/contour-deviation** differ from the milling run by design
  (flagged `⚙️ expected-diff` in §8).
- **⚠️ Chatter flag NOT absent:** `R310=1` *does* occur (51 rows, 15 slots) despite the
  non-milling description — see §17. `slot_chatter_cases_summary` will therefore still emit a
  `Chatter=1` branch. Reconcile with the domain owner whether this is a planned label or a bug.

## 4b. 🔎 Is any data clearly missing? (see §17)
- **12 signals truly gone** (renames excluded). The impactful ones are **`WCSPosition`** and
  **`ToolOrientation`** (HF geometry, ~5.2 M rows each in old) and the tool-compensation family
  **`cuttEdgeParam[u1,*]`**, plus `R328`, `actToolLength1[u1]`, `A_DBD|0`. No direct new-name equivalent.
- **Near-empty columns** in new: `Value_String` (0.02%) and `Nc_variable_name` (0.37%) — sparse by
  design (string/NC-variable rows only), **not** a data loss.
- **Completeness is good:** all metadata signals (`R319/R321/R330/actToolRadius/R310/actToolIdent`)
  and all four origins cover **every slot** — nothing silently missing per-slot.
- `ET200_Data` carries **3 channels** (`dataA1CH1/CH2`, `dataA2CH1`); confirm with the owner whether
  a 4th scope channel was expected.

## 5. 🛠️ Files & functions to refactor next
Grounded in a code search of `src/`, `viz/`, `data_cleaning/`:

| Priority | File | Function(s) | Change |
|---|---|---|---|
| 🔴 must | `src/loader.py` | `slot_metadata_summary` | `actToolIdent[u1,1]` ➜ `actToolIdent[u1]` (or resolve by stem) — else `Werkzeug` = NULL |
| 🔴 must | `src/data_processing.py` | `filter_constant_HF_signals`, `prepare_df_wide_for_pca`, `prepare_equal_bins_heatmap_sql`, `get_min_max_amplitudes_sql_from_db`, `get_min_max_amplitudes_sql` | default `target_origin='Oscilloscope'` ➜ `'ET200_Data'` |
| 🔴 must | `viz/widgets.py` | `update_heatmap`, `on_generate_clicked` (×2) | hardcoded `"Oscilloscope"` origin ➜ `"ET200_Data"` |
| 🟡 low | `src/data_processing.py` | `extract_unique_signal_values` | selects dropped `Label` ➜ KeyError, **but no callers exist** (latent). Fix only if used: `label_for(signal)`, or delete the helper |
| 🟡 should | `src/data_processing.py` | `filter_unique_gcodes` | `gcode_column='HFBlockEvent_GCode'` gone ➜ source from `HF_Event` (`HF_EVENT|NcCode`) |
| 🟡 should | `viz/visualizer.py` | `add_gcode_vrects`, `create_plot_with_gcode_annotations`, `create_axiswise_plots2` | `HFBlockEvent_GCode` + `HFProbeCounter` hover ➜ HF_Event-based |
| 🟢 safe | `src/loader.py` | `get_axiswise_plot_df` | already filters to existing cols — no crash; GCode overlay just empty until remapped |
| 🟢 verify | `src/loader.py` | `slot_chatter_cases_summary` | `R310` present **and has `Value=1` in 15 slots** — verify this flag is meaningful before trusting the `Chatter=1` branch |

**Recommended approach:** introduce a small **signal/origin resolver** (a dict or helper that maps
canonical names → the actual name in the connected schema) so the same code runs on both DBs
instead of scattering literal strings. Centralize `"Oscilloscope"/"ET200_Data"` as one constant.

## 6. Verdict
✅ **Safe to migrate.** Structure, plate/slot axes, origins (with the ET200 rename) and all LF
analysis signals line up, and every analysis-critical signal is complete per slot (§17). Before
switching production code: apply §5 (make signal/origin names schema-adaptive). **Two open items
for the domain owner:** (a) the `R310=1` chatter flag on a supposedly non-milling run, and (b) the
12 dropped signals — mainly `WCSPosition`/`ToolOrientation`/`cuttEdgeParam[u1,*]` — confirm they
are genuinely not needed downstream. Sections §10–§17 back all of this up.

---

# 🛠️ Refactoring Plan — migrating the codebase onto `DBnew`

**Objective:** make the whole project run correctly on `DBnew.duckdb` (table `cleaned_data`, ET200,
no *real* milling) **without breaking** compatibility with `DBold.duckdb`, by replacing scattered
hardcoded signal/column/origin literals with a single schema-adaptive layer.

**Guiding principles**
- 🎯 *Single source of truth*: one place maps canonical names → the name in the connected schema.
- 🔁 *Backward compatible*: the same code keeps working on `DBold` (needed for regression checks).
- 🤫 *No silent failures*: a canonical name that resolves to nothing should warn, not return NULL.
- ✅ *Validated by this notebook*: each phase is verified against the §10–§17 checks above.

> ### ⚠️ Design constraint — the dataset will GROW
> `DBnew` currently holds **only 1 plate (Platte 40) and 40 slots** *because this was an
> initial non-milling test run* — **more plates and slots will arrive later on this same schema**.
> What we migrate is the **schema**, not a fixed size. Therefore the refactor must:
>
> - **Never hardcode** plate/slot counts, `40`, or `Platte == 40`; keep all multi-plate /
>   multi-slot loops (`list_plates`, `list_slots_for_plate`, `plate_slots`) fully data-driven.
> - Keep `slot_metadata_summary`, heatmaps and per-slot logic **generic over N plates × M slots**.
> - Treat the single-plate/no-chatter facts as *properties of today's data*, not code assumptions.
> - In Phase 7, ideally re-validate on a future multi-plate `DBnew` drop, not just this one.

---

## Phase 0 · Baseline & safety net
- [ ] Branch off `new_schema` (e.g. `refactor/schema-adaptive`).
- [ ] Capture a **golden baseline** on `DBold`: run `slot_metadata_summary`,
      `slot_chatter_cases_summary`, and one heatmap/axiswise plot; save outputs to `results/`.
      These are the regression targets — behaviour on OLD must not change.
- [ ] Confirm `DBnew` + `DBold` both live in `data/` so `provider.list_databases()` sees both.

## Phase 1 · Central schema-adaptive layer  *(new file: `src/schema_map.py`)*
The root cause of every 🔴 breakage is literal strings. Introduce one resolver used everywhere.

```python
# src/schema_map.py
CANONICAL_ORIGINS = {
    "external": ["ET200_Data", "Oscilloscope"],   # new name first, old as fallback
    "hf": ["HF_Data"], "lf": ["LF_Data"], "hf_event": ["HF_Event"],
}
CANONICAL_SIGNALS = {
    "tool_ident": ["actToolIdent[u1]", "actToolIdent[u1,1]"],
    "tool_number":["actTNumber[u1]",  "actTNumber[u1,1]"],
    "nut_id": ["R319"], "x_pos": ["R321"], "speed": ["R330"],
    "tool_radius": ["actToolRadius[u1]"], "chatter": ["R310"],
}
def resolve_origin(loader, key) -> str | None: ...   # first candidate present in schema
def resolve_signal(loader, key) -> str | None: ...   # normalise whitespace, try fallbacks, warn if none

# German display names dropped with the old `Label` column (§17c) — kept in code, not the DB:
SIGNAL_LABELS = {
    "R301": "Synchronisationsstatus", "R310": "Rattererkennung",
    "R319": "Zähler für Nuten",        "R320": "Anzahl der zu fräsenden Nuten",
    "R321": "X-Position Nut",          "R322": "X-Versatz für nächste Nut",
    "R323": "Y-Position Nut",          "R328": "Z-Position Nut",   # signal itself dropped in new
    "R330": "Spindeldrehzahl",         "R331": "Drehzahländerung nach Nut",
    "R333": "Vorschub für Nutfräsen [mm/U]",
}
SCOPE_LABELS = {"dataA1CH1": "Kanal A (V)", "dataA2CH1": "Kanal B (V)"}  # A/B assignment: confirm w/ owner
```

- [ ] Implement `resolve_signal` with whitespace normalisation (`[u1,1]`↔`[u1, 1]`) and a
      channel-drop fallback (`[u1,1]`→`[u1]`), matching §5's rename detector.
- [ ] Resolver **warns** (not silently NULL) when nothing matches.
- [ ] Add `SIGNAL_LABELS`/`SCOPE_LABELS` + a `label_for(signal)` helper (German display names — §17c
      decision). These replace the dropped `Label` column and work on **both** DBs.
- [ ] Unit-test against **both** loaders (new→`ET200_Data`/`actToolIdent[u1]`, old→`Oscilloscope`/`actToolIdent[u1,1]`).

## Phase 2 · `src/loader.py`
- [ ] **`slot_metadata_summary`** — replace literal `'actToolIdent[u1,1]'` (+ the R319/R321/R330/
      actToolRadius block) with `resolve_signal(self, ...)`. Fixes `Werkzeug = NULL`.
      *Tool ident lives in `Value_String` (§16) — the fix is the **name**, not the column.*
- [ ] **`slot_chatter_cases_summary`** — resolve `R310`; keep the `Chatter=1` branch, but **flag the
      §17 discrepancy** (R310=1 on a non-milling run) until the domain owner confirms the label.
- [ ] **`get_axiswise_plot_df`** — already filters to existing cols (🟢). Swap dropped
      `HFBlockEvent_GCode`/`HFProbeCounter` for the HF_Event source (Phase 5).
- [ ] Treat `Platte`/`Nut` as `int` end-to-end; drop str assumptions. **Do NOT hardcode counts.**

## Phase 3 · `src/data_processing.py`
- [ ] `prepare_equal_bins_heatmap_sql`, `get_min_max_amplitudes_sql_from_db`,
      `get_min_max_amplitudes_sql`, `prepare_df_wide_for_pca`, `filter_constant_HF_signals` —
      replace default `target_origin='Oscilloscope'` with `resolve_origin(loader, "external")`.
- [ ] `filter_unique_gcodes` — default `gcode_column='HFBlockEvent_GCode'` gone; re-point to HF_Event (Phase 5).
- [ ] `extract_unique_signal_values` — **low priority / currently uncalled.** Selects the dropped
      `Label` column (latent KeyError). If kept, replace `Label` with `label_for(signal)` from
      `SIGNAL_LABELS` (German, §17c); otherwise delete the helper.

## Phase 4 · `viz/` (widgets & visualizer)
- [ ] **`viz/widgets.py`** — `update_heatmap`, `on_generate_clicked` (×2): the three hardcoded
      `"Oscilloscope"` origins → `resolve_origin(loader, "external")` / shared constant.
- [ ] **`viz/visualizer.py`** — `add_gcode_vrects`, `create_plot_with_gcode_annotations`,
      `create_axiswise_plots2`: `HFBlockEvent_GCode` + `HFProbeCounter` hover → HF_Event-based (Phase 5).

## Phase 5 · HF block-event / GCode remap  *(the structural change)*
Wide `HFBlockEvent_*` columns became the `HF_Event` **origin** (`HF_EVENT|NcCode`, `|NcComment`,
`|IpoGC`, `|SeekOffset`, `|Channel`, …).
- [ ] Add a loader helper (e.g. `get_hf_events(plate, slot)`) returning block events in the shape the
      GCode overlay expects (Time/Duration + GCode text).
- [ ] Map old→new: `HFBlockEvent_GCode`→`HF_EVENT|NcCode`/`NcComment`, `_Channel`→`HF_EVENT|Channel`,
      `_SeekOffset`→`HF_EVENT|SeekOffset`.
- [ ] Point `filter_unique_gcodes` and the visualizer overlays at this helper.
- [ ] `SensorType` gone → derive HF/accelerometer classification from `DataOrigin`.

## Phase 6 · Types, config & missing-signal decisions
- [ ] Centralise the external-origin constant (`"ET200_Data"`); `"Oscilloscope"` only in `schema_map.py`.
- [ ] Audit configs/notebooks for literal `"Oscilloscope"` / `my_table` / plate-as-string / hardcoded `40`.
- [ ] Verify `Time` (`TIMESTAMP`) still parses in `get_data_df`'s `pd.to_datetime`.
- [ ] **Decide on the 12 dropped signals** (§17): confirm `WCSPosition`/`ToolOrientation`/
      `cuttEdgeParam[u1,*]`/`R328`/`actToolLength1[u1]` are not needed, or request them in the next drop.

## Phase 7 · Validation / regression
- [ ] Re-run this notebook's §10–§17 — all green on `DBnew`.
- [ ] Re-run the Phase 0 golden baseline on `DBold` → outputs **identical** (no regression).
- [ ] On `DBnew`: `slot_metadata_summary(40)` returns non-null `Werkzeug`; one heatmap + one axiswise
      plot render (GCode overlay from HF_Event).
- [ ] Re-validate on a **future multi-plate** `DBnew` drop when it arrives (data-growth guardrail).

---

## Execution order & effort
| # | Phase | Files | Risk | Depends on |
|---|---|---|---|---|
| 1 | Resolver | `src/schema_map.py` (new) | low | — |
| 2 | Loader | `src/loader.py` | med | Phase 1 |
| 3 | Data processing | `src/data_processing.py` | med | Phase 1 |
| 4 | Viz origins | `viz/widgets.py`, `viz/visualizer.py` | low | Phase 1 |
| 5 | HF_Event remap | `loader.py`, `data_processing.py`, `visualizer.py` | **high** | Phase 2–4 |
| 6 | Types / config / missing-signal decisions | configs / notebooks | low | Phase 1–5 |
| 7 | Validation | this notebook + baseline | — | all |

**Suggested first PR:** Phases 1–2 (resolver + loader) — smallest change that fixes the `Werkzeug`
bug and makes lookups schema-adaptive, covered by §11/§17. Phase 5 is the only structural piece and
should be its own PR.